<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/team-elo.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · Team Race Elo ratings

Reproduce the analytics webapp's **Team Race Elo Ratings** feature: a chess-style
Elo rating per school, updated match-by-match in chronological order from every
team-racing regatta in a season.

Each school starts at **1500**. After every match, the winner takes rating points
from the loser — more if the win was against a stronger opponent or by a wide
margin, less if it was against a weak opponent or narrow. Regattas that matter
more (conference/national championships) move ratings further than routine
regular-season events.

This notebook reproduces the **core rating algorithm** compactly. The full,
authoritative implementation — including a week-of-season K-factor escalation,
raw penalty/DNF position backfill from a SQLite finish table, strength-of-schedule,
conference-adjusted ratings, and a "luck" stat — lives in
[`analytics/elo/engine.py`](https://github.com/TWalkerSMCM/icsa-tools) (745 lines) in
the private `icsa-tools` monorepo. Where this notebook simplifies something, it
says so and points at the lines to go read for the full treatment.

## Install

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"


## The Elo update rule

For a match between school A (rating $R_a$) and school B (rating $R_b$), the
expected win probability for A is the standard logistic Elo formula:

$$E_a = \frac{1}{1 + 10^{(R_b - R_a) / 400}}$$

After the match, A's actual result $S_a$ is `1.0` for a win, `0.0` for a loss.
**Ties take no rating change at all** — they're skipped entirely, same as the
engine (`engine.py` drops tied matchups before they ever reach the rating step).

The rating change combines a **K-factor** (how much this regatta matters) with a
**margin-of-victory multiplier** (how lopsided the match was), a log curve that
never *shrinks* K — it only ever equals or inflates it:

$$\text{mov} = \max\!\big(1.0,\ \ln(\text{margin} + 1) \times 1.5\big)$$

$$\Delta = K \times \text{mov} \times (S_a - E_a), \qquad R_a \mathrel{+}= \Delta,\ \ R_b \mathrel{-}= \Delta$$

`margin` is the difference between the two teams' summed finishing positions for
the race (each team-of-3 match's positions always sum to 21 combined, so a 1-2-3
sweep beats 4-5-6 by a margin of 12).

**K-factor** scales with how much the regatta matters. The full engine also steps
K up in the second half of the season (`_classify_k`, `engine.py` lines 36-49,
using `_first_saturday`/`_week_number` helpers at lines 54-90); this notebook
reproduces only the regatta-type tiers, using the engine's early-season values:

| regatta type contains… | K |
|---|---|
| "national championship" | 50 |
| "conference championship" | 40 |
| "cross regional" | 20 |
| "regional" (non-cross) | 15 |
| "fundamental" | 5 |
| anything else | 10 |

## Filters (mirroring the engine)

- **Multi-team exclusion**: when a school enters more than one squad in a
  regatta (A/B teams), only its best-placed squad counts. The engine picks by
  `dt_rank` from a SQLite `team` table (lines 224-252); this library's
  `TeamRaceTeam.place` is the equivalent signal, so we group by school and keep
  the lowest `place`. *Caveat*: a `TeamRaceMatch.opponent` is recorded as a
  school slug, not a specific opposing squad, so unlike the engine (which checks
  **both** sides are best-squads via team IDs) we can only guarantee **our**
  side is the best squad. This only matters for the rare regatta where a school
  fields multiple squads.
- **Ties**: excluded from rating updates entirely (see above).
- **Unsailed matches** (`sailed=False`, scheduled but not yet raced): skipped.
- **Chronological order**: sorted by `regatta_start`, then `race_num` — the
  engine sorts by the Monday of the match's calendar week, then regatta name,
  then race number (lines 376-380); ordering within a single regatta by
  `race_num` is equivalent for our purposes since one regatta's matches all
  share a `regatta_start`.
- **DNF/DSQ/RAF position backfill**: not needed here. The engine reconstructs
  1-6 placements from raw penalty codes in a SQLite `finish` table
  (`_map_penalty_positions`, lines 93-147) because it works from unprocessed
  race data. This library's `TeamRaceMatch.our_positions` / `their_positions`
  and `won` / `tied` are already fully resolved by the parser, so we use them
  directly.

## Load the season's team-racing regattas

`sailors=False` skips per-sailor pages we don't need for team-race Elo.
`.team()` narrows the dataset to team-scoring regattas only. The first run
scrapes and caches to disk (`./.scraper_cache`); re-runs are instant.

In [ ]:
import scraper

SEASON = "s26"
data = scraper.load(SEASON, sailors=False).team()
print(len(data.regattas), "team-racing regattas")

## Build one match record per race

Each `TeamRaceMatch` appears twice in the raw data — once in each team's own
`rounds[].matches[]` — so we dedupe to a single record per
`(season, slug, race_num)` by keeping only the lexicographically-smaller
school's perspective. We also apply the best-squad-per-school filter and pull
each regatta's `regatta_type` (when captured) for the K-factor lookup.

In [ ]:
school_name = {}
for reg in data.regattas:
    for team in reg.teams:
        school_name.setdefault(team.school_slug, team.school)

# regatta_type per (season, slug), when overview metadata was captured during load
_rt = data.regattas_frame()
regatta_type_by_key = dict(zip(zip(_rt["season"], _rt["slug"]), _rt["regatta_type"]))


def classify_k(regatta_type):
    """Regatta-importance tiers from engine.py's `_classify_k`, without the
    week-of-season escalation (see markdown above)."""
    t = (regatta_type or "").lower()
    if "national championship" in t:
        return 50
    if "conference championship" in t:
        return 40
    if "cross regional" in t:
        return 20
    if "fundamental" in t:
        return 5
    if "regional" in t:
        return 15
    return 10


def best_squads(regatta):
    """One team per school: the lowest (best) `place` — mirrors the engine's
    dt_rank-based multi-team exclusion."""
    best = {}
    for team in regatta.teams:
        if team.school_slug not in best or team.place < best[team.school_slug].place:
            best[team.school_slug] = team
    return best


matches = []
for reg in data.regattas:
    best = best_squads(reg)
    k = classify_k(regatta_type_by_key.get((reg.season, reg.slug), ""))
    seen = set()
    for slug, team in best.items():
        for rnd in team.rounds:
            for m in rnd.matches:
                if not m.sailed or slug > m.opponent:
                    continue  # unsailed, or the opponent's perspective (dedupe)
                key = (reg.season, reg.slug, m.race_num, slug, m.opponent)
                if key in seen:
                    continue
                seen.add(key)
                matches.append(
                    {
                        "season": reg.season,
                        "slug": reg.slug,
                        "regatta_start": reg.regatta_start,
                        "race_num": m.race_num,
                        "team_a": slug,
                        "team_b": m.opponent,
                        "tied": m.tied,
                        "a_won": m.won,
                        "score_a": sum(m.our_positions),
                        "score_b": sum(m.their_positions),
                        "k": k,
                    }
                )

matches.sort(key=lambda m: (m["regatta_start"], m["slug"], m["race_num"]))
print(
    len(matches),
    "deduplicated matches across",
    len({(m["season"], m["slug"]) for m in matches}),
    "regattas",
)

## Run the Elo computation

Walk the matches in chronological order, updating both schools' ratings after
each one. Ties and unsailed matches never reach this step (filtered above).

In [ ]:
import math
from collections import defaultdict

INITIAL_RATING = 1500.0


def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))


def mov_multiplier(margin):
    return max(1.0, math.log(abs(margin) + 1) * 1.5)


ratings = defaultdict(lambda: INITIAL_RATING)
wins = defaultdict(int)
losses = defaultdict(int)
ties = defaultdict(int)
history = defaultdict(list)  # school_slug -> [(regatta_start, rating), ...]

for m in matches:
    a, b = m["team_a"], m["team_b"]
    if m["tied"]:
        ties[a] += 1
        ties[b] += 1
        continue

    ra, rb = ratings[a], ratings[b]
    margin = abs(m["score_a"] - m["score_b"])
    expected_a = expected_score(ra, rb)
    actual_a = 1.0 if m["a_won"] else 0.0
    delta = m["k"] * mov_multiplier(margin) * (actual_a - expected_a)

    ratings[a] = ra + delta
    ratings[b] = rb - delta
    if m["a_won"]:
        wins[a] += 1
        losses[b] += 1
    else:
        wins[b] += 1
        losses[a] += 1

    history[a].append((m["regatta_start"], ratings[a]))
    history[b].append((m["regatta_start"], ratings[b]))

print(len(ratings), "schools rated")

## Final ratings table

In [ ]:
import pandas as pd

ratings_df = (
    pd.DataFrame(
        [
            {
                "school_slug": slug,
                "school": school_name.get(slug, slug),
                "elo": round(rating, 1),
                "wins": wins[slug],
                "losses": losses[slug],
                "ties": ties[slug],
            }
            for slug, rating in ratings.items()
        ]
    )
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)
ratings_df.index = range(1, len(ratings_df) + 1)
ratings_df.head(20)

## Ratings over time — top schools

In [ ]:
import matplotlib.pyplot as plt

TOP_N_CHART = 8
top_slugs = ratings_df.head(TOP_N_CHART)["school_slug"].tolist()

plt.figure(figsize=(9, 6))
for slug in top_slugs:
    dates = pd.to_datetime([d for d, _ in history[slug]])
    values = [v for _, v in history[slug]]
    plt.plot(dates, values, marker="o", markersize=3, label=school_name.get(slug, slug))

plt.axhline(INITIAL_RATING, color="gray", linestyle="--", linewidth=0.8)
plt.ylabel("Elo rating")
plt.title(f"{SEASON} Team Race Elo — top {TOP_N_CHART} schools over time")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

---
**To go further:** this notebook rates one season in isolation, starting every
school back at 1500. The production engine also computes, on top of the same
core rating loop — all in `analytics/elo/engine.py`, cited by line above and
below:

- week-of-season K escalation (lines 36-90)
- strength-of-schedule (mean opponent rating at time of match)
- conference-adjusted rating: `elo + (conference_strength - 1500) * 0.5` (lines 639-644)
- a "luck" stat: `sum(actual - expected)` across a school's matches (lines 622-628)
- separate coed/women's categories, driven from `reg.participant` in the source DB

Multi-season carryover (e.g. `scraper.load(["f25", "s26"])`) works the same way
as here — just don't reset `ratings` between seasons.